# latex-task3 · Macrocyclic-peptide (MCP) table → LaTeX

Converts the FuncBind MCP table in `results/reports/results_mcp.html`.

## Setup

In [ ]:
import sys
from pathlib import Path
def _find_repo_root():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "voxbind" / "dataset").is_dir():
            return c
    fb = Path("/home/shpark/prj-denovo/VoxBind")
    if (fb / "voxbind" / "dataset").is_dir():
        return fb
    raise FileNotFoundError("repo root (voxbind/dataset) not found from cwd")
_REPO = _find_repo_root()
sys.path.insert(0, str(_REPO / "notebook" / "results"))
from bs4 import BeautifulSoup, Tag
from latex_common import *   # shared HTML->LaTeX helpers + generic table_to_latex

MCP_HTML_PATH = Path("results/reports/results_mcp.html")


## MCP converter + output

In [ ]:
# MCP 결과 전용 소스: results_mcp.html
MCP_HTML_PATH = Path("results/reports/results_mcp.html")

# 긴 헤더/모델명은 표에 맞게 줄여 씁니다 (없는 키는 원문을 escape 처리).
MCP_HEADER_RENAME = {
    "Molecules":   r"$n$",
    "High aff.":   r"High aff.\ \%",
    "Diversity":   r"Div.",
    "Cyclization": r"Cycl.",
}
MCP_MODEL_RENAME = {
    "FuncBind vanilla":         r"FuncBind (vanilla)",
    "FuncBind + receptor-ED":   r"FuncBind + receptor-ED",
    "Reference cyclic peptide": r"Reference",
}


def mcp_generation_to_latex(table: Tag) -> str:
    """Macrocyclic-peptide generation table (results_mcp.html) -> LaTeX.

    One header row, one data row per model. The Vina Dock cell already carries an
    "avg / med" pair, kept as a single column to match the source. A row whose body
    collapses to one spanning cell (a not-yet-run setting, e.g. the full-budget
    placeholder) is emitted commented out so the template keeps its shape; PoseCheck
    is still TBA and prints as-is. A \midrule is inserted before the Reference row.
    """
    def fix(s: str) -> str:
        s = s.replace("−", "-").replace("—", "---").replace("–", "--")
        return s.replace("&", r"\&").replace("%", r"\%").replace("#", r"\#")

    def header_label(th: Tag) -> str:
        c = th.__copy__()
        sub = c.find(class_="sub")
        sub_txt = sub.get_text(" ", strip=True) if sub else ""
        if sub:
            sub.decompose()
        main = c.get_text(" ", strip=True)
        arrow = (r" $\downarrow$" if ("↓" in sub_txt or "↓" in main)
                 else r" $\uparrow$" if ("↑" in sub_txt or "↑" in main) else "")
        return MCP_HEADER_RENAME.get(main, escape_latex_text(main)) + arrow

    def model_label(td: Tag) -> str:
        c = td.__copy__()
        sub = c.find(class_="sub")
        if sub:
            sub.decompose()
        name = c.get_text(" ", strip=True)
        return MCP_MODEL_RENAME.get(name, escape_latex_text(name))

    labels = [header_label(th) for th in table.find("thead").find_all("th")]
    colspec = "l" + "c" * (len(labels) - 1)

    lines, n_hidden = [], 0
    for tr in table.find("tbody").find_all("tr"):
        cells = tr.find_all(["th", "td"], recursive=False)
        if any(int(c.get("colspan", 1)) > 1 for c in cells):   # spanning placeholder
            n_hidden += 1
            vals = [model_label(cells[0])] + [fix(c.get_text(" ", strip=True)) for c in cells[1:]]
            lines.append("% " + " & ".join(vals) + r" \\")
            continue
        name = model_label(cells[0])
        vals = [fix(c.get_text(" ", strip=True)) for c in cells[1:]]
        if name == "Reference" and lines and lines[-1] != r"\midrule":
            lines.append(r"\midrule")
        lines.append(" & ".join([name, *vals]) + r" \\")

    title_p = table.find_previous("p", class_="table-title")
    title = title_p.get_text(" ", strip=True) if title_p else ""
    title = title.split("·", 1)[-1].strip() if "·" in title else title
    caption = (r"\textbf{Macrocyclic-peptide generation with FuncBind.} "
               + escape_latex_text(title)
               + r". Vina Dock is reported as avg\,/\,med over the generated peptides, "
                 r"High aff.\ is the share out-docking the reference crystal peptide, and "
                 r"$n$ is the number of peptides kept of the 250 requested per pocket. "
                 r"Cyclization is the fraction of valid macrocycles; PoseCheck is still "
                 r"being computed.")

    def ind(rows, level):
        return [INDENT * level + r for r in rows]

    header = [rf"\begin{{tabular}}{{{colspec}}}", r"\toprule",
              " & ".join(labels) + r" \\", r"\midrule"]
    out = ([r"\begin{table}[t]"]
           + ind([r"\centering",
                  r"\caption{" + caption + "}",
                  r"\label{tab:mcp-generation}",
                  r"\resizebox{.98\textwidth}{!}{%"], 1)
           + ind(header[:1], 2)
           + ind([*header[1:], *lines, r"\bottomrule"], 3)
           + ind([r"\end{tabular}"], 2)
           + ind([r"}"], 1)
           + (ind([f"% {n_hidden} row(s) commented out: not yet run at build time."], 1)
              if n_hidden else [])
           + [r"\end{table}"])
    return "\n".join(out)


mcp_path = resolve_html_path(MCP_HTML_PATH)
mcp_soup = BeautifulSoup(mcp_path.read_text(encoding="utf-8"), "html.parser")
mcp_table = mcp_soup.find("table")
if mcp_table is None:
    raise LookupError(f"{mcp_path} 에 표가 없습니다.")

print(f"Source: {mcp_path}")
print(mcp_generation_to_latex(mcp_table))
